# Day 2 — Prime Numbers
**Student:** Md. Shishir Kaysar Shuvon  
**Student ID:** `IT-23030`  
**Department:** ICT, MBSTU

## Scope
This notebook follows the Day 2 lecture on prime numbers: infinitude and distribution of primes, Dirichlet's theorem, prime number theorem, conjectures, prime generation, primality testing, sieve of Eratosthenes, pseudoprimes, Carmichael numbers, primitive roots, Lucas tests, Mersenne primes, perfect numbers, antiprimes and the fundamental theorem of arithmetic.

Some lecture topics are major theorems or open conjectures, not short programming exercises. Those are explained as theory; computationally suitable topics are implemented.

In [ ]:
import math

def gcd(a, b):
    a, b = abs(a), abs(b)
    while b:
        a, b = b, a % b
    return a

def euler_phi(n):
    if n < 1:
        raise ValueError('n must be positive')
    result, x, p = n, n, 2
    while p*p <= x:
        if x % p == 0:
            while x % p == 0:
                x //= p
            result -= result // p
        p += 1
    if x > 1:
        result -= result // x
    return result


: 

## 1. Prime numbers

A prime is a natural number greater than 1 that is not a product of two smaller natural numbers. The first primes are `2, 3, 5, 7, 11, 13, ...`. The number 1 is not prime.

In [ ]:
def is_prime_trial(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    d = 3
    while d*d <= n:
        if n % d == 0:
            return False
        d += 2
    return True

print([(n, is_prime_trial(n)) for n in range(1, 16)])

Trial division only needs to check divisors up to `√n`: if `n=ab` is composite, at least one of `a,b` is no greater than `√n`. The implementation skips even divisors after checking 2.

**Complexity:** `O(√n)` trial divisions in the worst case, with `O(1)` extra space.

## 2. Infinitude of primes — Euclid's idea

There are infinitely many primes. A classic proof assumes a finite list `p1,...,pk` of all primes and considers

\[
N=p_1p_2\cdots p_k+1.
\]

No listed prime divides `N`, because division by any `pi` leaves remainder 1. But every integer greater than 1 has a prime divisor. This contradicts the claim that the list contained all primes.

The lecture also mentions other proofs, including an argument involving the divergence of the harmonic series and topological ideas. These are theoretical results rather than required coding tasks.

## 3. Prime distribution, Dirichlet's theorem and Prime Number Theorem

**Dirichlet's theorem (lecture statement):** if natural numbers `a,b` are coprime, there are infinitely many primes of the form `ak+b`. The lecture notes that its proof is not elementary and belongs to analytic number theory.

Let `π(x)` count primes up to `x`. The **Prime Number Theorem** states asymptotically

\[
\pi(x)\sim\frac{x}{\log x}.
\]

This is an asymptotic estimate, not an exact equality for every `x`.

In [ ]:
def prime_count_upto(n):
    return sum(is_prime_trial(k) for k in range(2, n+1))

for n in [100, 1000, 10000]:
    actual = prime_count_upto(n)
    estimate = n / math.log(n)
    print(f"x={n:5d}, pi(x)={actual:4d}, x/log(x)≈{estimate:.2f}")

## 4. Prime conjectures and polynomial examples

The lecture introduces several famous questions:
- **Goldbach conjecture:** every even integer greater than 3 is the sum of two primes.
- **Twin prime conjecture:** there are infinitely many prime pairs differing by 2.
- **Waring's prime number conjecture** (as named in the slides): every odd number greater than 3 is either prime or a sum of three primes.
- It also mentions Chebyshev's bias, primes between consecutive squares, and prime spirals.

These are conjectures or topics of research; code that checks a finite range cannot prove an infinite statement.

The polynomial `n²+n+41` gives prime values for `n=0,...,39`, but not for all integers.

In [ ]:
def goldbach_pair(even_n):
    if even_n <= 3 or even_n % 2:
        return None
    for p in range(2, even_n):
        if is_prime_trial(p) and is_prime_trial(even_n-p):
            return p, even_n-p
    return None

print("Goldbach pair for 100:", goldbach_pair(100))
print("n^2+n+41 for n=0..5:", [n*n+n+41 for n in range(6)])
print("At n=40:", 40*40+40+41, "(composite)")

## 5. Sieve of Eratosthenes

To find all primes up to `n`, initially mark every integer at least 2 as prime. For each prime `p` up to `√n`, cross out its multiples starting at `p²`. The remaining marked numbers are prime.

In [ ]:
def sieve_of_eratosthenes(n):
    if n < 2:
        return []
    prime = [True] * (n + 1)
    prime[0] = prime[1] = False
    p = 2
    while p*p <= n:
        if prime[p]:
            for multiple in range(p*p, n+1, p):
                prime[multiple] = False
        p += 1
    return [i for i, flag in enumerate(prime) if flag]

print(sieve_of_eratosthenes(50))

**Complexity:** `O(n log log n)` time and `O(n)` space. The lecture also mentions wheel factorization, including the wheel of 30, which skips candidates divisible by 2, 3 or 5.

## 6. Fermat primality test, pseudoprimes and Carmichael numbers

Fermat's Little Theorem says that for prime `p` and `gcd(a,p)=1`,

\[
a^{p-1}\equiv1\pmod p.
\]

If the congruence fails, `n` is composite. If it passes, `n` may still be composite; such a composite is a Fermat pseudoprime to base `a`.

A **Carmichael number** is composite but satisfies `a^n ≡ a (mod n)` for every integer `a`. The first examples listed in the lecture include `561, 1105, 1729, 2465, 2821`.

In [ ]:
def fermat_probable_prime(n, bases=(2, 3, 5)):
    if n < 2:
        return False
    if n in (2, 3):
        return True
    for a in bases:
        if gcd(a, n) != 1:
            return False
        if pow(a, n-1, n) != 1:
            return False
    return True

for n in [7, 91, 341, 561, 1105]:
    print(n, "trial prime:", is_prime_trial(n),
          "| passes selected Fermat bases:", fermat_probable_prime(n))

**Important:** a probable-prime result is not a proof. Carmichael numbers demonstrate why Fermat testing alone is insufficient. The code above uses a few fixed bases for demonstration and must not be used as a cryptographic primality guarantee.

## 7. Primitive roots, order and modular structure

For a modulus `n`, the multiplicative order of `a` (when `gcd(a,n)=1`) is the smallest positive integer `k` such that

\[
a^k\equiv1\pmod n.
\]

A primitive root modulo `n` is a residue whose order equals `φ(n)`. For a prime `p`, the nonzero residues modulo `p` form a multiplicative group of size `p−1`; hence `φ(p)=p−1`.

In [ ]:
def multiplicative_order(a, n):
    if gcd(a, n) != 1:
        return None
    value = 1
    for k in range(1, euler_phi(n)+1):
        value = (value * a) % n
        if value == 1:
            return k
    return None

def is_primitive_root(a, n):
    return multiplicative_order(a, n) == euler_phi(n)

print("Order of 2 mod 7:", multiplicative_order(2, 7))
print("Is 3 a primitive root mod 7?", is_primitive_root(3, 7))
print("Primitive roots mod 7:", [a for a in range(1,7) if is_primitive_root(a,7)])

## 8. Lucas theorem and n−1 primality tests

The lecture states a Lucas primality criterion: if `n>1` and for every prime factor `q` of `n−1` there is an integer `a` such that

\[
a^{n-1}\equiv1\pmod n,
\qquad
a^{(n-1)/q}\not\equiv1\pmod n,
\]

then `n` is prime. The converse is not true in the form stated in the slides.

The slides also introduce the `n−1` test and name Pepin's test, Proth's theorem, Lucas–Lehmer test and `n+1` tests. These are advanced primality-testing methods; the present notebook does not claim to implement all of them.

## 9. Lucas–Lehmer test and Mersenne primes

A Mersenne number has the form

\[
M_p=2^p-1.
\]

When `M_p` is prime, it is called a Mersenne prime; `p` must itself be prime. The lecture gives the Lucas–Lehmer sequence

\[
s_0=4,\qquad s_{i+1}=s_i^2-2.
\]

For prime `p>2`, the Lucas–Lehmer criterion says `M_p` is prime exactly when `s_{p-2} ≡ 0 (mod M_p)`.

In [ ]:
def lucas_lehmer(p):
    if p == 2:
        return True
    if p < 2 or not is_prime_trial(p):
        return False
    M = (1 << p) - 1
    s = 4
    for _ in range(p-2):
        s = (s*s - 2) % M
    return s == 0

for p in [2, 3, 5, 7, 11, 13, 17, 19]:
    print(f"p={p:2d}, M_p={(1<<p)-1:7d}, Mersenne prime? {lucas_lehmer(p)}")

## 10. Perfect numbers

A positive integer is perfect if it equals the sum of its positive divisors excluding itself. The first examples in the lecture are `6, 28, 496, 8128`.

For an even perfect number, the classical Euclid–Euler result connects it with a Mersenne prime:
\[
2^{p-1}(2^p-1)
\]
is perfect whenever `2^p−1` is prime.

In [ ]:
def proper_divisor_sum(n):
    if n < 1:
        raise ValueError("n must be positive")
    total = 0
    for d in range(1, n):
        if n % d == 0:
            total += d
    return total

def is_perfect(n):
    return n > 1 and proper_divisor_sum(n) == n

print([(n, proper_divisor_sum(n), is_perfect(n)) for n in [6, 28, 496, 8128]])

## 11. Antiprimes (highly composite numbers)

The lecture defines an antiprime / highly composite number as a number with more divisors than any smaller positive integer. If

\[
n=p_1^{a_1}p_2^{a_2}\cdots p_k^{a_k},
\]

then the number of positive divisors is

\[
d(n)=\prod_{i=1}^{k}(a_i+1).
\]

In [ ]:
def divisor_count(n):
    if n < 1:
        raise ValueError("n must be positive")
    count = 0
    for d in range(1, math.isqrt(n)+1):
        if n % d == 0:
            count += 1 if d*d == n else 2
    return count

def highly_composite_numbers(limit):
    result = []
    best = 0
    for n in range(1, limit+1):
        count = divisor_count(n)
        if count > best:
            best = count
            result.append((n, count))
    return result

print("First highly composite numbers up to 100:")
print(highly_composite_numbers(100))
print("d(60) =", divisor_count(60))

## 12. Fundamental Theorem of Arithmetic and applications

The Fundamental Theorem of Arithmetic states that every integer greater than 1 can be expressed as a product of primes, uniquely up to the order of the factors.

The lecture connects primes with number-theoretic functions, modular arithmetic, cryptography and algebraic structures (including rings). It notes that multiplying two integers is easy compared with factoring their product in general; this asymmetry motivates RSA-style public-key cryptography. It also mentions quadratic reciprocity, cyclotomic polynomials and the Riemann Hypothesis as further topics.

In [ ]:
def prime_factorization(n):
    if n < 1:
        raise ValueError("n must be positive")
    factors = []
    d = 2
    while d*d <= n:
        exponent = 0
        while n % d == 0:
            n //= d
            exponent += 1
        if exponent:
            factors.append((d, exponent))
        d += 1
    if n > 1:
        factors.append((n, 1))
    return factors

for n in [60, 84, 360, 2024]:
    print(n, "=", prime_factorization(n))

## Conclusion

This notebook combines mathematical statements from the Day 2 lecture with small computational demonstrations. Finite experiments illustrate patterns but do not prove conjectures such as Goldbach's or the twin-prime conjecture. Advanced criteria are described at the level presented in the slides; only selected algorithms are implemented here.

## References
- Course resource: **Day 2.pdf**, supplied with the assignment.